## 📦 Step 1: Install Dependencies & Load Dataset

In [ ]:
# Install required libraries
!pip install kagglehub xgboost --quiet

In [ ]:
import kagglehub
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Download dataset
path = kagglehub.dataset_download("yasserh/titanic-dataset")
print("Path to dataset files:", path)

# List files in the downloaded path
for f in os.listdir(path):
    print(" -", f)

In [ ]:
# Load the CSV file
csv_file = [f for f in os.listdir(path) if f.endswith('.csv')][0]
df = pd.read_csv(os.path.join(path, csv_file))

print(f"Dataset Shape: {df.shape}")
df.head(10)

## 🔍 Step 2: Exploratory Data Analysis (EDA)

In [ ]:
# Basic dataset info
print("=" * 50)
print("DATASET INFO")
print("=" * 50)
df.info()

In [ ]:
# Missing values summary
print("=" * 50)
print("MISSING VALUES")
print("=" * 50)
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])

In [ ]:
# Statistical summary
print("=" * 50)
print("STATISTICAL SUMMARY")
print("=" * 50)
df.describe()

In [ ]:
# ── EDA Visualizations ──
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Titanic Dataset - Exploratory Data Analysis', fontsize=18, fontweight='bold', y=1.01)

# 1. Survival Count
survival_counts = df['Survived'].value_counts()
axes[0, 0].bar(['Did Not Survive (0)', 'Survived (1)'], survival_counts.values,
               color=['#e74c3c', '#2ecc71'], edgecolor='white', linewidth=1.5)
axes[0, 0].set_title('Survival Count', fontweight='bold')
axes[0, 0].set_ylabel('Number of Passengers')
for i, v in enumerate(survival_counts.values):
    axes[0, 0].text(i, v + 5, str(v), ha='center', fontweight='bold')

# 2. Survival by Gender
gender_survival = df.groupby(['Sex', 'Survived']).size().unstack()
gender_survival.plot(kind='bar', ax=axes[0, 1], color=['#e74c3c', '#2ecc71'],
                     edgecolor='white', linewidth=1.5)
axes[0, 1].set_title('Survival by Gender', fontweight='bold')
axes[0, 1].set_xlabel('Gender')
axes[0, 1].set_ylabel('Count')
axes[0, 1].legend(['Did Not Survive', 'Survived'])
axes[0, 1].tick_params(axis='x', rotation=0)

# 3. Survival by Passenger Class
pclass_survival = df.groupby(['Pclass', 'Survived']).size().unstack()
pclass_survival.plot(kind='bar', ax=axes[0, 2], color=['#e74c3c', '#2ecc71'],
                     edgecolor='white', linewidth=1.5)
axes[0, 2].set_title('Survival by Passenger Class', fontweight='bold')
axes[0, 2].set_xlabel('Passenger Class')
axes[0, 2].set_ylabel('Count')
axes[0, 2].legend(['Did Not Survive', 'Survived'])
axes[0, 2].tick_params(axis='x', rotation=0)

# 4. Age Distribution by Survival
survived = df[df['Survived'] == 1]['Age'].dropna()
not_survived = df[df['Survived'] == 0]['Age'].dropna()
axes[1, 0].hist(not_survived, bins=30, alpha=0.7, color='#e74c3c', label='Did Not Survive')
axes[1, 0].hist(survived, bins=30, alpha=0.7, color='#2ecc71', label='Survived')
axes[1, 0].set_title('Age Distribution by Survival', fontweight='bold')
axes[1, 0].set_xlabel('Age')
axes[1, 0].set_ylabel('Count')
axes[1, 0].legend()

# 5. Fare Distribution by Survival
survived_fare = df[df['Survived'] == 1]['Fare'].dropna()
not_survived_fare = df[df['Survived'] == 0]['Fare'].dropna()
axes[1, 1].hist(not_survived_fare, bins=40, alpha=0.7, color='#e74c3c', label='Did Not Survive')
axes[1, 1].hist(survived_fare, bins=40, alpha=0.7, color='#2ecc71', label='Survived')
axes[1, 1].set_title('Fare Distribution by Survival', fontweight='bold')
axes[1, 1].set_xlabel('Fare')
axes[1, 1].set_ylabel('Count')
axes[1, 1].legend()

# 6. Survival Rate by Embarked Port
embarked_survival = df.groupby('Embarked')['Survived'].mean() * 100
colors = ['#3498db', '#9b59b6', '#e67e22']
embarked_survival.plot(kind='bar', ax=axes[1, 2], color=colors, edgecolor='white')
axes[1, 2].set_title('Survival Rate (%) by Embarkation Port', fontweight='bold')
axes[1, 2].set_xlabel('Port (C=Cherbourg, Q=Queenstown, S=Southampton)')
axes[1, 2].set_ylabel('Survival Rate (%)')
axes[1, 2].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('eda_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print("EDA plots saved!")

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(10, 7))
numeric_df = df.select_dtypes(include=[np.number])
corr_matrix = numeric_df.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn',
            mask=mask, linewidths=0.5, cbar_kws={"shrink": .8})
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 🛠️ Step 3: Data Preprocessing & Feature Engineering

In [ ]:
# Work on a copy
df_processed = df.copy()

# ── Feature Engineering ──

# 1. Extract Title from Name
df_processed['Title'] = df_processed['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
title_map = {
    'Mr': 'Mr', 'Miss': 'Miss', 'Mrs': 'Mrs', 'Master': 'Master',
    'Dr': 'Rare', 'Rev': 'Rare', 'Col': 'Rare', 'Major': 'Rare',
    'Mlle': 'Miss', 'Countess': 'Rare', 'Ms': 'Miss', 'Lady': 'Rare',
    'Jonkheer': 'Rare', 'Don': 'Rare', 'Dona': 'Rare', 'Mme': 'Mrs',
    'Capt': 'Rare', 'Sir': 'Rare'
}
df_processed['Title'] = df_processed['Title'].map(title_map).fillna('Rare')
print("Titles found:", df_processed['Title'].value_counts().to_dict())

# 2. Family Size
df_processed['FamilySize'] = df_processed['SibSp'] + df_processed['Parch'] + 1
df_processed['IsAlone'] = (df_processed['FamilySize'] == 1).astype(int)

# 3. Age Bands
df_processed['Age'].fillna(df_processed.groupby(['Pclass', 'Sex'])['Age'].transform('median'), inplace=True)
df_processed['AgeBand'] = pd.cut(df_processed['Age'],
                                  bins=[0, 12, 18, 35, 60, 100],
                                  labels=['Child', 'Teen', 'YoungAdult', 'Adult', 'Senior'])

# 4. Fare Bands
df_processed['Fare'].fillna(df_processed['Fare'].median(), inplace=True)
df_processed['FareBand'] = pd.qcut(df_processed['Fare'], q=4,
                                    labels=['Low', 'Medium', 'High', 'VeryHigh'])

# 5. Cabin Indicator
df_processed['HasCabin'] = df_processed['Cabin'].notna().astype(int)

# 6. Embarked – fill missing with mode
df_processed['Embarked'].fillna(df_processed['Embarked'].mode()[0], inplace=True)

print("\nFeature Engineering Complete!")
print(df_processed[['Title', 'FamilySize', 'IsAlone', 'AgeBand', 'FareBand', 'HasCabin']].head(10))

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Select features for the model
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare',
            'Embarked', 'Title', 'FamilySize', 'IsAlone', 'HasCabin',
            'AgeBand', 'FareBand']

df_model = df_processed[features + ['Survived']].copy()

# Encode categorical columns
categorical_cols = ['Sex', 'Embarked', 'Title', 'AgeBand', 'FareBand']
le = LabelEncoder()
for col in categorical_cols:
    df_model[col] = le.fit_transform(df_model[col].astype(str))

print("Processed feature matrix shape:", df_model.shape)
print("\nMissing values after processing:")
print(df_model.isnull().sum())
df_model.head()

## 🤖 Step 4: Model Training

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, roc_auc_score, roc_curve)

# Split features and target
X = df_model.drop('Survived', axis=1)
y = df_model['Survived']

# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features (useful for Logistic Regression, SVM, KNN)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set size : {X_train.shape[0]} samples")
print(f"Test set size     : {X_test.shape[0]} samples")

# ── Define Models ──
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=200, random_state=42),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=200, random_state=42),
    'XGBoost':             XGBClassifier(n_estimators=200, use_label_encoder=False,
                                         eval_metric='logloss', random_state=42),
    'SVM':                 SVC(probability=True, random_state=42),
    'KNN':                 KNeighborsClassifier(n_neighbors=7)
}

# ── Train & Evaluate ──
results = {}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("\n" + "=" * 65)
print(f"{'Model':<25} {'CV Accuracy':>12} {'Test Accuracy':>14} {'ROC-AUC':>10}")
print("=" * 65)

for name, model in models.items():
    # Use scaled data for LR, SVM, KNN
    if name in ['Logistic Regression', 'SVM', 'KNN']:
        X_tr, X_te = X_train_scaled, X_test_scaled
    else:
        X_tr, X_te = X_train, X_test

    # Cross-validation
    cv_scores = cross_val_score(model, X_tr, y_train, cv=cv, scoring='accuracy')

    # Train & predict
    model.fit(X_tr, y_train)
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1]

    acc  = accuracy_score(y_test, y_pred)
    auc  = roc_auc_score(y_test, y_prob)

    results[name] = {
        'cv_mean': cv_scores.mean(), 'cv_std': cv_scores.std(),
        'test_acc': acc, 'roc_auc': auc,
        'y_pred': y_pred, 'y_prob': y_prob, 'model': model
    }

    print(f"{name:<25} {cv_scores.mean():.4f} ± {cv_scores.std():.3f}   {acc:.4f}         {auc:.4f}")

print("=" * 65)

## 📊 Step 5: Model Evaluation & Comparison

In [ ]:
# ── Model Comparison Bar Chart ──
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Model Performance Comparison', fontsize=15, fontweight='bold')

model_names = list(results.keys())
test_accs   = [results[m]['test_acc'] for m in model_names]
roc_aucs    = [results[m]['roc_auc']  for m in model_names]

colors = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6', '#f39c12', '#1abc9c']

# Test Accuracy
bars = axes[0].bar(model_names, test_accs, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Test Accuracy', fontweight='bold')
axes[0].set_ylim(0.7, 1.0)
axes[0].set_ylabel('Accuracy')
axes[0].tick_params(axis='x', rotation=30)
for bar, val in zip(bars, test_accs):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                 f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

# ROC-AUC
bars2 = axes[1].bar(model_names, roc_aucs, color=colors, edgecolor='white', linewidth=1.5)
axes[1].set_title('ROC-AUC Score', fontweight='bold')
axes[1].set_ylim(0.7, 1.0)
axes[1].set_ylabel('AUC')
axes[1].tick_params(axis='x', rotation=30)
for bar, val in zip(bars2, roc_aucs):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                 f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ── ROC Curves ──
plt.figure(figsize=(9, 7))
colors_roc = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6', '#f39c12', '#1abc9c']

for (name, res), color in zip(results.items(), colors_roc):
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    plt.plot(fpr, tpr, color=color, lw=2,
             label=f"{name} (AUC = {res['roc_auc']:.3f})")

plt.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Classifier')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves – All Models', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=9)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Best Model – Detailed Evaluation ──
best_model_name = max(results, key=lambda k: results[k]['roc_auc'])
best = results[best_model_name]

print(f"🏆 Best Model: {best_model_name}")
print(f"   Test Accuracy : {best['test_acc']:.4f}")
print(f"   ROC-AUC       : {best['roc_auc']:.4f}")
print()
print("Classification Report:")
print("=" * 55)
print(classification_report(y_test, best['y_pred'],
                             target_names=['Did Not Survive', 'Survived']))

# Confusion Matrix
fig, ax = plt.subplots(figsize=(7, 5))
cm = confusion_matrix(y_test, best['y_pred'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Did Not Survive', 'Survived'],
            yticklabels=['Did Not Survive', 'Survived'],
            linewidths=1, linecolor='white')
ax.set_title(f'Confusion Matrix – {best_model_name}', fontweight='bold', fontsize=13)
ax.set_xlabel('Predicted Label', fontsize=11)
ax.set_ylabel('True Label', fontsize=11)
plt.tight_layout()
plt.show()

## 🌲 Step 6: Feature Importance

In [ ]:
# Use Random Forest for feature importance
rf_model = results['Random Forest']['model']
feature_names = X.columns.tolist()
importances = rf_model.feature_importances_
fi_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
fi_df = fi_df.sort_values('Importance', ascending=True)

plt.figure(figsize=(9, 7))
colors_fi = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(fi_df)))
plt.barh(fi_df['Feature'], fi_df['Importance'], color=colors_fi, edgecolor='white')
plt.xlabel('Feature Importance (Gini)', fontsize=12)
plt.title('Feature Importance – Random Forest', fontsize=14, fontweight='bold')
plt.axvline(x=fi_df['Importance'].mean(), color='red', linestyle='--',
            alpha=0.7, label=f'Mean = {fi_df["Importance"].mean():.3f}')
plt.legend()
plt.tight_layout()
plt.show()

print("\nTop 5 Most Important Features:")
print(fi_df.sort_values('Importance', ascending=False).head(5).to_string(index=False))

## 🎯 Step 7: Hyperparameter Tuning (Best Model)

In [ ]:
from sklearn.model_selection import GridSearchCV

print("Tuning Random Forest hyperparameters...")

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth':    [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf':  [1, 2]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train, y_train)

print("\nBest Parameters:", grid_search.best_params_)
print(f"Best CV ROC-AUC: {grid_search.best_score_:.4f}")

# Evaluate tuned model
best_rf = grid_search.best_estimator_
y_pred_tuned = best_rf.predict(X_test)
y_prob_tuned = best_rf.predict_proba(X_test)[:, 1]

print(f"\nTuned Model Test Accuracy : {accuracy_score(y_test, y_pred_tuned):.4f}")
print(f"Tuned Model ROC-AUC       : {roc_auc_score(y_test, y_prob_tuned):.4f}")

## 🔮 Step 8: Predict on a New Passenger

In [ ]:
# Example: Predict survival for a new passenger
# Features: Pclass, Sex, Age, SibSp, Parch, Fare, Embarked,
#           Title, FamilySize, IsAlone, HasCabin, AgeBand, FareBand

# Encoding reference (LabelEncoder was fit on training data)
# We'll manually encode for demonstration

new_passengers = pd.DataFrame({
    'Pclass':     [3, 1, 2],
    'Sex':        [1, 0, 0],      # 1=male, 0=female (label encoded)
    'Age':        [22, 35, 28],
    'SibSp':      [1, 0, 0],
    'Parch':      [0, 0, 1],
    'Fare':       [7.25, 71.0, 30.0],
    'Embarked':   [2, 0, 2],      # 2=S, 0=C, 1=Q
    'Title':      [2, 2, 1],      # 2=Mr, 1=Miss, 0=Master, 3=Mrs
    'FamilySize': [2, 1, 2],
    'IsAlone':    [0, 1, 0],
    'HasCabin':   [0, 1, 0],
    'AgeBand':    [2, 2, 2],      # 2=YoungAdult
    'FareBand':   [0, 3, 1]       # 0=Low, 1=Medium, 3=VeryHigh
})

passenger_labels = [
    'Passenger A (3rd class male, age 22)',
    'Passenger B (1st class female, age 35)',
    'Passenger C (2nd class female, age 28)'
]

predictions = best_rf.predict(new_passengers)
probabilities = best_rf.predict_proba(new_passengers)[:, 1]

print("\n" + "=" * 55)
print("SURVIVAL PREDICTIONS FOR NEW PASSENGERS")
print("=" * 55)
for label, pred, prob in zip(passenger_labels, predictions, probabilities):
    status = "✅ SURVIVED" if pred == 1 else "❌ DID NOT SURVIVE"
    print(f"{label}")
    print(f"  Prediction : {status}")
    print(f"  Probability of Survival: {prob:.1%}")
    print()

## 📋 Step 9: Summary

In [ ]:
print("=" * 60)
print("          📊 PROJECT SUMMARY – TITANIC SURVIVAL PREDICTION")
print("=" * 60)
print(f"\n  Dataset Size     : {df.shape[0]} passengers, {df.shape[1]} features")
print(f"  Survival Rate    : {df['Survived'].mean():.1%}")
print(f"  Features Used    : {len(X.columns)}")

print("\n  Model Leaderboard (by ROC-AUC):")
print(f"  {'Model':<25} {'Test Acc':>10} {'ROC-AUC':>10}")
print("  " + "-" * 47)
sorted_models = sorted(results.items(), key=lambda x: x[1]['roc_auc'], reverse=True)
for rank, (name, res) in enumerate(sorted_models, 1):
    marker = " 🏆" if rank == 1 else "   "
    print(f"{marker} {name:<25} {res['test_acc']:.4f}     {res['roc_auc']:.4f}")

print(f"\n  ✅ Best Model (after tuning): Random Forest")
print(f"     Test Accuracy : {accuracy_score(y_test, y_pred_tuned):.4f}")
print(f"     ROC-AUC       : {roc_auc_score(y_test, y_prob_tuned):.4f}")
print("\n  Key Insights:")
print("   • Gender (Sex) is the strongest survival predictor")
print("   • Passenger class (Pclass) strongly correlates with survival")
print("   • Fare paid reflects socioeconomic status and survival odds")
print("   • Title extracted from name is a powerful engineered feature")
print("=" * 60)